# 235. Lowest Common Ancestor of a Binary Search Tree
**Difficulty:** 🟡 Medium · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-search-tree/

## 💡 Concepts

**Core concept(s):** Use the **BST order rule** to walk straight to the split point.

**Why it applies here:** The lowest common ancestor is the first node where the two targets go different ways (one smaller, one larger) — or a node equal to one of them. The BST ordering tells you which way to walk, so you never explore wrong branches.

**Key intuition:** If both targets are smaller, go left; if both larger, go right; otherwise you're standing on the split — that's the answer.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is a Binary Search Tree (BST)?
A **BST** stays ordered: for every node, everything in its **left** subtree is smaller and everything in its **right** subtree is larger — so you can find values by going left/right like binary search.
- **Key fact:** an **in-order** walk of a BST visits the values in **sorted** order.

---

**Prerequisite knowledge:**
- The BST ordering property.

## 📝 Problem

Given a BST and two values `p`, `q`, return the value of their lowest common ancestor (the deepest node that has both in its subtrees).

**Example**
```
tree = [6,2,8,0,4,7,9], p=2, q=8 -> 6
tree = [6,2,8,0,4,7,9], p=2, q=4 -> 2
```

> Two approaches, both `O(h)`: iterative walk and recursion.

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Iterative Walk (optimal)

**Idea:** From the root, if both targets are smaller go left; if both larger go right; otherwise this node is the split point (the LCA).

**Time:** `O(h)`. **Space:** `O(1)`.

In [ ]:
def lca_iter(root: Optional[TreeNode], p: int, q: int) -> int:
    node = root
    while node:
        if p < node.val and q < node.val:  # both targets are smaller -> go left
            node = node.left
        elif p > node.val and q > node.val:# both targets are larger -> go right
            node = node.right
        else:                              # they split here (one each side, or equal)
            return node.val                # this is the lowest common ancestor
    return -1

### Approach 2 — Recursion

**Idea:** Same rule, expressed recursively.

**Time:** `O(h)`. **Space:** `O(h)`.

In [ ]:
def lca_rec(root: Optional[TreeNode], p: int, q: int) -> int:
    if p < root.val and q < root.val:      # both smaller -> answer is in the left subtree
        return lca_rec(root.left, p, q)
    if p > root.val and q > root.val:      # both larger -> answer is in the right subtree
        return lca_rec(root.right, p, q)
    return root.val                        # the split point is the lowest common ancestor

In [ ]:
# Correctness check
root = build_tree([6,2,8,0,4,7,9])
tests = [(2,8,6), (2,4,2), (7,9,8), (0,4,2)]
for p, q, exp in tests:
    a, b = lca_iter(root, p, q), lca_rec(root, p, q)
    print(f"LCA({p},{q}) -> iter={a}, rec={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_balanced(n), 1, n)   # extremes -> walk down to a split
solutions = {
    "iterative O(h)": lca_iter,
    "recursive O(h)": lca_rec,
}
sizes = [2000, 4000, 8000, 16000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Exploit BST ordering to prune:** you never look at a branch that can't contain the answer — that's why it's `O(h)`, not `O(n)`.
- **LCA = the split point:** the first node where the two targets diverge.
- **Signal:** "lowest/least common ancestor in a BST", "where do two paths meet".
- **Related problems:** LCA of a Binary Tree (no BST order — harder), Insert/Delete in a BST.
- **Common pitfalls:** (1) ignoring the case where a target equals the current node; (2) forgetting the BST version is much simpler than the general tree version.